In [8]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

# =========================
# 1) LOAD MONTHLY RETURNS
# =========================
returns = pd.read_csv(
    "../data/processed/monthly_returns.csv",
    index_col=0,
    parse_dates=True
).sort_index()

# =========================
# 2) PARAMETERS
# =========================
WINDOW = 120          # 10 years * 12 months
MIN_OBS = 36          # at least 3 years of monthly returns
STALE_THRESHOLD = 0.5
START_YEAR = 2013
END_YEAR = 2024
RIDGE = 1e-8          # smaller regularization for monthly returns

# =========================
# 3) MIN VAR SOLVER
# =========================
def solve_min_variance(cov_matrix):
    n = cov_matrix.shape[0]
    Sigma = cov_matrix.values

    def objective(w):
        return w.T @ Sigma @ w

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    bounds = [(0, 1)] * n
    x0 = np.ones(n) / n

    res = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-12}
    )

    if not res.success:
        raise ValueError(f"Optimization failed: {res.message}")

    w = pd.Series(res.x, index=cov_matrix.index)
    w[w.abs() < 1e-12] = 0.0
    w = w / w.sum()

    return w

# =========================
# 4) STORAGE
# =========================
all_weights = []
all_portfolio_returns = []

# =========================
# 5) ANNUAL REBALANCING
# =========================
for year in range(START_YEAR, END_YEAR + 1):

    dec_date = returns.index[
        (returns.index.year == year) &
        (returns.index.month == 12)
    ]

    if len(dec_date) == 0:
        print(f"No December date for {year}, skipped")
        continue

    dec_date = dec_date[-1]
    next_year = year + 1

    # 10-year estimation window up to Dec Y
    window_data = returns.loc[:dec_date].tail(WINDOW)

    if len(window_data) < WINDOW:
        print(f"{year}: not enough window data")
        continue

    # keep assets with at least 36 monthly observations
    valid_assets = window_data.columns[window_data.count() >= MIN_OBS]
    window_data = window_data[valid_assets]

    # exclude stale prices: too many zero monthly returns
    zero_ratio = window_data.eq(0).sum() / window_data.count()
    window_data = window_data.loc[:, zero_ratio <= STALE_THRESHOLD]

    # exclude assets with zero or near-zero volatility
    window_data = window_data.loc[:, window_data.std(skipna=True) > 1e-8]

    if window_data.shape[1] < 2:
        print(f"{year}: less than 2 valid assets")
        continue

    # covariance matrix using available pairwise observations
    cov_Y = window_data.cov()

    # drop assets still creating NaN covariance rows/cols
    cov_Y = cov_Y.dropna(axis=0, how="any").dropna(axis=1, how="any")
    common_assets = cov_Y.index.intersection(cov_Y.columns)
    cov_Y = cov_Y.loc[common_assets, common_assets]

    if cov_Y.shape[0] < 2:
        print(f"{year}: covariance matrix too small")
        continue

    # small ridge to improve numerical stability
    cov_Y = cov_Y + RIDGE * np.eye(len(cov_Y))

    # =========================
    # OPTIMIZATION AT END OF YEAR Y
    # =========================
    try:
        w_rebalanced = solve_min_variance(cov_Y)
    except Exception as e:
        print(f"{year}: optimization failed - {e}")
        continue

    all_weights.append(pd.DataFrame({
        "RebalanceDate": dec_date,
        "Year": year,
        "ISIN": w_rebalanced.index,
        "Weight": w_rebalanced.values
    }))

    print(
        f"{year}: optimized with {len(w_rebalanced)} assets | "
        f"sum weights = {w_rebalanced.sum():.6f} | "
        f"max weight = {w_rebalanced.max():.4f}"
    )

    # =========================
    # APPLY WEIGHTS DURING YEAR Y+1
    # =========================
    next_returns = returns.loc[
        returns.index.year == next_year,
        w_rebalanced.index
    ]

    if next_returns.empty:
        print(f"{year}: no returns for {next_year}")
        continue

    current_weights = w_rebalanced.copy()

    for dt, r_t in next_returns.iterrows():

        # Important: do NOT renormalize by dropping missing returns too aggressively.
        # If a return is missing, set it to 0 only if you consider it a missing report.
        # Delistings should ideally already be handled before this stage as -100%.
        r_t = r_t.reindex(current_weights.index)

        if r_t.isna().all():
            continue

        r_t = r_t.fillna(0.0)

        # portfolio return for month t
        rp_t = (current_weights * r_t).sum()

        all_portfolio_returns.append({
            "Date": dt,
            "RebalanceYear": year,
            "Return": rp_t
        })

        # update weights within the year according to Lecture 5
        if 1 + rp_t <= 0:
            print(f"Portfolio collapsed at {dt}")
            break

        current_weights = current_weights * (1 + r_t) / (1 + rp_t)

        # numerical cleanup
        current_weights[current_weights.abs() < 1e-12] = 0.0
        current_weights = current_weights / current_weights.sum()

# =========================
# 6) EXPORT
# =========================
weights_df = pd.concat(all_weights, ignore_index=True)

returns_df = pd.DataFrame(all_portfolio_returns)
returns_df["Date"] = pd.to_datetime(returns_df["Date"])
returns_df = returns_df.sort_values("Date")

returns_df["Cumulative"] = (1 + returns_df["Return"]).cumprod() - 1

weights_df.to_csv("../data/processed/mv_weights.csv", index=False)
returns_df.to_csv("../data/processed/mv_portfolio_returns.csv", index=False)

print("MVP monthly + annual rebalancing terminé ✅")
print("Weights saved to ../data/processed/mv_weights.csv")
print("Returns saved to ../data/processed/mv_portfolio_returns.csv")

2013: optimized with 494 assets | sum weights = 1.000000 | max weight = 0.1762
2014: optimized with 494 assets | sum weights = 1.000000 | max weight = 0.1543
2015: optimized with 492 assets | sum weights = 1.000000 | max weight = 0.2225
2016: optimized with 494 assets | sum weights = 1.000000 | max weight = 0.2129
2017: optimized with 495 assets | sum weights = 1.000000 | max weight = 0.1800
2018: optimized with 495 assets | sum weights = 1.000000 | max weight = 0.3722
2019: optimized with 495 assets | sum weights = 1.000000 | max weight = 0.3235
2020: optimized with 494 assets | sum weights = 1.000000 | max weight = 0.2394
2021: optimized with 495 assets | sum weights = 1.000000 | max weight = 0.2176
2022: optimized with 498 assets | sum weights = 1.000000 | max weight = 0.1588
2023: optimized with 500 assets | sum weights = 1.000000 | max weight = 0.1964
2024: optimized with 499 assets | sum weights = 1.000000 | max weight = 0.1445
MVP monthly + annual rebalancing terminé ✅
Weights s

In [ ]:
import pandas as pd
mv_weights = pd.read_csv(
    "../data/processed/mv_weights.csv",
    index_col=0,
    parse_dates=True
).sort_index()
mv_returns = pd.read_csv(
    "../data/processed/mv_portfolio_returns.csv",
    index_col=0,
    parse_dates=True
).sort_index()
returns = pd.read_csv(
    "../data/processed/Returns.csv",
    index_col=0,
    parse_dates=True
).sort_index()

In [14]:
# =========================
# TOP 3 WEIGHTS PAR ANNÉE
# =========================

weights_df["RebalanceDate"] = pd.to_datetime(weights_df["RebalanceDate"])

top3_weights_by_year = (
    weights_df
    .sort_values(["Year", "Weight"], ascending=[True, False])
    .groupby("Year")
    .head(3)
)

print("\n========== TOP 3 WEIGHTS PAR ANNÉE ==========\n")

for year, group in top3_weights_by_year.groupby("Year"):
    print(f"\nYEAR {year}")
    print(group[["ISIN", "Weight"]].to_string(index=False))


# =========================
# TOP 3 WEIGHTS GLOBAUX
# =========================

top3_weights_global = (
    weights_df
    .sort_values("Weight", ascending=False)
    .head(3)
)

print("\n========== TOP 3 WEIGHTS GLOBAUX ==========\n")
print(top3_weights_global[["Year", "ISIN", "Weight"]].to_string(index=False))


# =========================
# TOP 3 RETURNS PAR ANNÉE
# =========================

returns_df["Date"] = pd.to_datetime(returns_df["Date"])
returns_df["Year"] = returns_df["Date"].dt.year

top3_returns_by_year = (
    returns_df
    .sort_values(["Year", "Return"], ascending=[True, False])
    .groupby("Year")
    .head(3)
)

print("\n========== TOP 3 RETURNS PAR ANNÉE ==========\n")

for year, group in top3_returns_by_year.groupby("Year"):
    print(f"\nYEAR {year}")
    print(
        group[["Date", "Return", "Cumulative"]]
        .to_string(index=False)
    )


# =========================
# TOP 3 RETURNS GLOBAUX
# =========================

top3_returns_global = (
    returns_df
    .sort_values("Return", ascending=False)
    .head(3)
)

print("\n========== TOP 3 RETURNS GLOBAUX ==========\n")

print(
    top3_returns_global[
        ["Date", "Return", "Cumulative"]
    ].to_string(index=False)
)


========== TOP 3 WEIGHTS PAR ANNÉE ==========


YEAR 2013
        ISIN   Weight
HK0006000050 0.176241
JP3414750004 0.077969
HK0002007356 0.077806

YEAR 2014
        ISIN   Weight
HK0002007356 0.154326
JP3951200009 0.094369
JP3982100004 0.067994

YEAR 2015
        ISIN   Weight
HK0002007356 0.222501
JP3982100004 0.078651
JP3814800003 0.056123

YEAR 2016
        ISIN   Weight
HK0002007356 0.212941
JP3188220002 0.114388
JP3711200000 0.092859

YEAR 2017
        ISIN   Weight
HK0002007356 0.180038
JP3711200000 0.112552
JP3789000001 0.104022

YEAR 2018
        ISIN   Weight
HK0002007356 0.372229
JP3982100004 0.127821
JP3400900001 0.044072

YEAR 2019
        ISIN   Weight
HK0002007356 0.323529
JP3982100004 0.101865
JP3326410002 0.054348

YEAR 2020
        ISIN   Weight
HK0002007356 0.239418
JP3982100004 0.115223
JP3613000003 0.112748

YEAR 2021
        ISIN   Weight
HK0002007356 0.217636
JP3982100004 0.100963
JP3613000003 0.098968

YEAR 2022
        ISIN   Weight
HK0002007356 0.158825
JP3901

In [ ]:
# =========================
# PORTFOLIO PERFORMANCE METRICS
# =========================

# monthly returns
rp = returns_df["Return"]

# annualized average return
annual_return = rp.mean() * 12

# annualized volatility
annual_volatility = rp.std() * np.sqrt(12)

# Sharpe ratio (risk-free assumed = 0)
sharpe_ratio = annual_return / annual_volatility

# cumulative return over full sample
cumulative_return = (1 + rp).prod() - 1

# minimum / maximum monthly return
min_return = rp.min()
max_return = rp.max()

# =========================
# PRINT RESULTS
# =========================

print("\n========== PORTFOLIO PERFORMANCE ==========\n")

print(f"Annualized Return      : {annual_return:.4%}")
print(f"Annualized Volatility  : {annual_volatility:.4%}")
print(f"Sharpe Ratio           : {sharpe_ratio:.4f}")

print(f"\nCumulative Return      : {cumulative_return:.4%}")

print(f"\nMin Monthly Return     : {min_return:.4%}")
print(f"Max Monthly Return     : {max_return:.4%}")


========== PORTFOLIO PERFORMANCE ==========

Annualized Return      : 7.2694%
Annualized Volatility  : 9.8902%
Sharpe Ratio           : 0.7350

Cumulative Return      : 125.2262%

Min Monthly Return     : -6.3076%
Max Monthly Return     : 8.6824%


In [11]:
# =========================
# LOAD RF
# =========================

rf_df = pd.read_csv("../data/processed/rf_rate.csv")

# rename columns
rf_df.columns = ["Date", "RF"]

# YYYYMM -> datetime
rf_df["Date"] = pd.to_datetime(
    rf_df["Date"].astype(str),
    format="%Y%m"
)

# move to month-end
rf_df["Date"] = rf_df["Date"] + pd.offsets.MonthEnd(0)

# convert percentage to decimal
rf_df["RF"] = rf_df["RF"] / 100

print(rf_df.head())
# =========================
# PORTFOLIO PERFORMANCE METRICS
# =========================

returns_df["Date"] = pd.to_datetime(returns_df["Date"])

# merge portfolio returns + RF
perf_df = returns_df.merge(rf_df, on="Date", how="left")

# monthly portfolio returns
rp = perf_df["Return"]

# monthly risk-free rate
rf = perf_df["RF"]

# excess monthly returns
excess_returns = rp - rf

# annualized average return
annual_return = rp.mean() * 12

# annualized average RF
annual_rf = rf.mean() * 12

# annualized excess return
annual_excess_return = excess_returns.mean() * 12

# annualized volatility
annual_volatility = rp.std() * np.sqrt(12)

# Sharpe ratio
sharpe_ratio = annual_excess_return / annual_volatility

# cumulative return over full sample
cumulative_return = (1 + rp).prod() - 1

# minimum / maximum monthly return
min_return = rp.min()
max_return = rp.max()

# =========================
# PRINT RESULTS
# =========================

print("\n========== PORTFOLIO PERFORMANCE ==========\n")

print(f"Annualized Return      : {annual_return:.4%}")
print(f"Annualized RF          : {annual_rf:.4%}")
print(f"Annualized Volatility  : {annual_volatility:.4%}")

print(f"\nSharpe Ratio           : {sharpe_ratio:.4f}")

print(f"\nCumulative Return      : {cumulative_return:.4%}")

print(f"\nMin Monthly Return     : {min_return:.4%}")
print(f"Max Monthly Return     : {max_return:.4%}")

        Date      RF
0 2000-01-31  0.0041
1 2000-02-29  0.0043
2 2000-03-31  0.0047
3 2000-04-30  0.0046
4 2000-05-31  0.0050

========== PORTFOLIO PERFORMANCE ==========

Annualized Return      : 7.2694%
Annualized RF          : 1.7475%
Annualized Volatility  : 9.8902%

Sharpe Ratio           : 0.5583

Cumulative Return      : 125.2267%

Min Monthly Return     : -6.3078%
Max Monthly Return     : 8.6824%
